# [System Design Interview](./system-design.md)  
This session’s content can likely be asked when discussing System Design in interviews.    
You need to understand which solution fits which scenario and requirements. 

<img src='./pic/2_system_design.jpeg' width=650>

The 9-Phase System Design Framework

## PHASE 1: Problem Clarification & Scope (5 minutes)

**MOST IMPORTANT**: Clarify functional & non-functional requirements   
- What to focus (MVP features)
- What can be skip for now

**Must-Ask Questions**:
- **Features**: What's MVP vs future features?
- **Scale**: How many users? (1K, 100K, 1M, 100M?)
- **Usage Pattern**: Read-heavy or write-heavy?
- **Real-time Need**: Real-time updates or eventual consistency?
- **Data Loss Tolerance**: Can we lose 1 hour of data? Forever?

**Example**:

Interviewer: Design Instagram   

You: Let me clarify:    
- MVP: Upload photos, follow users, see feed?   
- Scale: 100M users, 50M DAU?   
- Read-heavy (viewing) or write-heavy (uploading)?   
- Real-time feed updates needed?    
- Can we lose uploaded photos? No? Need replication.     




## PHASE 2: Capacity Estimation (5 minutes)

**Calculate**:
- DAU (Daily Active Users)
- RPS (Requests Per Second)
- Data volume (Storage)

**Common Reference Numbers**:
- 1 server handles ~1,000 RPS
- At 600 RPS → Need 2-3 servers minimum
- Single database fine until 100M+ data rows (M: million)
- Storage estimate: (Users × Data per user per day × 365 × Years)

**Example**:

50M DAU, each user views 20 photos/day    
= 50M × 20 = 1B photo views/day      
= 1B / 86,400 seconds ≈ 11,600 RPS       
→ Need ~12-15 servers for photo serving       

megabytes (MB)   
megabits (Mb)



## PHASE 3: High-Level Architecture (10 minutes)
(call back the cache, top5 caching patterns, and message queue)    

**Always Include**:
- Load Balancer
- App Servers
- Cache Layer
- Database
- Message Queue (if necessary)

**Basic Architecture**:
```text
[User] → [DNS] → [Load Balancer] 
                     ↓
              [App Servers x3]
                     ↓
         ┌───────────┼───────────┐
         ↓           ↓           ↓
    [Cache]     [Database]   [Queue]
```



## ⭐️ PHASE 4: Data Model & Schema (10 minutes)
This is the key part. Better to know database design/ UML design.  

**MUST HAVE**:
- SQL schema with proper data types
- Primary keys (id, UUID)
- Indexes (on frequently queried columns)
- Foreign keys (relationships)
- Relationship types (1-1, 1-many, many-many)

**Example**:
```sql
CREATE TABLE users (
    id BIGINT PRIMARY KEY AUTO_INCREMENT,
    username VARCHAR(50) UNIQUE NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    created_at TIMESTAMP DEFAULT NOW(),
    INDEX idx_username (username),
    INDEX idx_email (email)
);
```



## PHASE 5: API Design (5 minutes)

**MUST-HAVE Endpoints**:
- Restful or
- GraphQL (most likely not needed)

**API Decisions**:  
- RESTful style (most common)
- Request & Response structure in JSON
- Pagination: Use `limit` + `offset` or cursor
- Error codes if asked
- Rate limiting if needed (e.g. 5 requests/second, for reliability of the system)

**Example**:
```html
POST   /api/v1/posts
GET    /api/v1/posts/{id}
GET    /api/v1/feed?limit=20&cursor=xyz
POST   /api/v1/users/{id}/follow
DELETE /api/v1/users/{id}/follow
GET    /api/v1/users/{id}/followers?limit=50&offset=0
```

**Response Structure**:
```json
{
  "data": {
    "posts": [...],
    "cursor": "next_page_token"
  },
  "meta": {
    "total": 1000,
    "limit": 20
  }
}
```

## PHASE 6: Caching Strategy (5 minutes)

**Use [Cache Decision Tree](#cache-decision-tree)** (from earlier section)

**Most Common**: Cache-Aside pattern
- Cache user profiles
- Cache news feeds
- Cache popular posts

```python
# Cache user profile (changes infrequently)
def get_user(user_id):
    cached = redis.get(f'user:{user_id}')
    if cached:
        return cached
    user = db.query(user_id)
    redis.set(f'user:{user_id}', user, ttl=3600)
    return user
```

## PHASE 7: Scaling Strategies (10 minutes)

**Scenario 1: Traffic Increases 10x**
- Add more app servers (horizontal scaling)
- Add cache layer for frequent queries
- Add message queue to offload heavy work

**Scenario 2: Database Too Slow**
- Add indexes on frequently queried columns
- Cache frequent queries
- Add read replicas (for read-heavy workloads)
- Eventually: Shard by user_id

**Scenario 3: One Server Dies**
- Load balancer health checks
- Automatically stops sending traffic to dead server
- Auto-scaling groups provision new server

**Scenario 4: Real-Time Updates Needed**
- Use WebSockets for push notifications
- Or use Server-Sent Events (SSE) for one-way updates



## 💡 PHASE 8: Monitoring & Alerting (5 minutes)
This differs you with other developers.  

**MUST-HAVE Metrics**:

**1. API Response Time**
- Track P50, P95, P99 latencies
- Alert if P95 > 500ms

**2. Request Throughput**
- Track RPS
- Alert if drops suddenly (outage)
- Alert if spikes beyond capacity

**3. Error Rate**
- Alert if > 3% errors
- Track 4xx (client errors) vs 5xx (server errors)

**4. Resource Metrics**
- CPU usage (alert if > 80%)
- Memory usage
- Disk space

**5. Database Metrics**
- Query latency
- Connection pool usage
- Slow query log

**6. Cache Metrics**
- Hit ratio (should be > 80%)
- Eviction rate

**Monitoring Tools**:
- Metrics: Datadog, New Relic, Prometheus
- Logging: Datadog, Splunk, ELK Stack
- Alerting: PagerDuty, Opsgenie



## PHASE 9: Trade-offs Discussion (5 minutes)

**For Each Major Decision**:
```
Option A:
Pros: X, Y, Z
Cons: A, B, C
When to use: [scenario]

Option B:
Pros: X, Y, Z
Cons: A, B, C
When to use: [scenario]
```

**Common Trade-Offs**:

**1. Consistency vs Availability (CAP Theorem)**
- Strong consistency: All nodes see same data (SQL)
- Eventual consistency: Nodes sync eventually (NoSQL)

**2. SQL vs NoSQL**
- SQL: Structured, ACID, relations
- NoSQL: Flexible schema, horizontal scaling

**3. REST vs GraphQL**
- REST: Simple, standard, cacheable
- GraphQL: Flexible queries, single endpoint

**4. Synchronous vs Asynchronous**
- Sync: Immediate response, simpler
- Async: Better performance, more complex



## Quick Decision Matrix

| Decision | Default Choice | Why |
|----------|----------------|-----|
| Database | SQL (PostgreSQL) | Simple, relational, scalable to 100M+ |
| Cache | Redis | Fast, simple, TTL support, many data structures |
| API | REST | Easiest to explain, industry standard |
| Load Balancer | Round-robin | Works well for most cases |
| Message Queue | Skip unless needed | Don't over-engineer for smaller scales |
| Sharding | Skip | Only needed at 100M+ users |
| Microservices | Skip | Too complex for interview scope |
| CDN | Mention briefly | For static files if global users |



##  Final Interview Checklist

Before ending the interview, ensure you've covered:

- ✅ Asked clarifying questions
- ✅ Drew a clear diagram
- ✅ Discussed performance/scalability
- ✅ Mentioned trade-offs
- ✅ Considered edge cases
- ✅ Thought about monitoring

---



# Concept Review

<img src='./pic/2_concepts.gif' width=700>

1️⃣ Load Balancing

Distribute traffic so no single server becomes the villain.
Without it → one node crashes, your app crashes.


2️⃣ Caching

Store hot data in memory.
Because hitting the database for everything is self-sabotage.


3️⃣ CDN (Content Delivery Network)

Static files served from the closest geographic edge.
Latency drops. Users smile. 🌍


4️⃣ Message Queue

Decouple producers and consumers.
Your system stops being synchronous and starts being resilient.


5️⃣ Publish–Subscribe

One event → many consumers.
Perfect for notifications, analytics, auditing, and event-driven systems.


6️⃣ API Gateway

Single entry point.
Routing. Auth. Rate limiting. Protocol translation.
Your microservices need a front door.


7️⃣ Circuit Breaker

When downstream services fail repeatedly, stop calling them.
Fail fast > Fail forever.


8️⃣ Service Discovery

Services come and go.
Your system must find them dynamically.


9️⃣ Sharding

Split large datasets across nodes using a shard key.
Because vertical scaling eventually dies.


🔟 Rate Limiting

Protect your system from abuse (or accidental traffic spikes).
No limits = self-DDoS.


1️⃣1️⃣ Consistent Hashing

Distribute data with minimal reshuffling when nodes change.
Critical for distributed caches and storage systems.


1️⃣2️⃣ Auto Scaling

Traffic up? Add servers.
Traffic down? Remove servers.
Elasticity = cost efficiency.


💡 Here’s the truth most people miss:

Individually, these patterns are simple.
Together, they create systems that survive Black Friday traffic, viral launches, and production chaos.

If you’re preparing for:
• System Design interviews
• Senior backend roles
• Architect positions

You should be comfortable explaining:
 • When to use each pattern
 • Trade-offs involved
 • Failure modes
 • Real-world examples
